In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import AzureOpenAI
import gradio as gr

# Load environment variables
load_dotenv()

In [ ]:
# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

MODEL_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")

# Print a message to confirm initialization
print("Azure OpenAI client initialized with model:", MODEL_NAME)

In [ ]:
# define functions that the model can call

def get_weather(location: str, unit: str = "fahrenheit") -> str:
    """Simulate weather API call"""
    return f"Weather in {location}: 84°{unit[0].upper()}, Sunny"


def calculate(operation: str, a: float, b: float) -> float:
    """Perform calculations"""
    operations = {
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide": lambda x, y: x / y if y != 0 else None,
    }
    return operations[operation](a, b)


# test 
print(get_weather("New York"))
print(calculate("add", 5, 3))

In [ ]:
# Create a handler that executes tool calls from the model

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}, arguments: {arguments}", flush=True)
        tool = globals().get(tool_name) 
        result = tool(**arguments) if tool else {} #
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
# Define tools/functions: 
# This is the JSON schema that describes the functions that the model can call. 
# The model will use this information to decide when to call a function and how to format the arguments.
# This format is based on the OpenAPI specs.

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the weather for a specific location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform basic arithmetic calculations",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "The arithmetic operation",
                    },
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"},
                },
                "required": ["operation", "a", "b"],
            },
        },
    },
]

# print
tools

In [ ]:
# Now we have everything set up - the client, the model, the tools, and the handler to execute tool calls.
# Let's create a chat loop that allows the model to call tools as needed.

def chat(message, history):
    messages = history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # This is the call to the LLM - see that we pass in the tools json
        response = client.chat.completions.create(model=MODEL_NAME, messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # If the LLM wants to call a tool, we do that!
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [ ]:
# Launch a Gradio interface to test the chat function
gr.ChatInterface(chat, type="messages").launch()